In [1]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score
from dataset import ChestXrayDataset
from tqdm import tqdm
import torch.nn as nn

# --- CONFIG ---
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "../models/swin1_best.pth"  # <--- SWIN

LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
    'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]

def evaluate_model():
    print(f"✅ Evaluating Swin Transformer on device: {DEVICE}")

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_dataset = ChestXrayDataset(
        data_dir=r'C:\Users\srbuh\Desktop\Medical_AI_Diagnosis\data\images',
        csv_file=r'C:\Users\srbuh\Desktop\Medical_AI_Diagnosis\data\Data_Entry_2017.csv',
        split_list_file=r'C:\Users\srbuh\Desktop\Medical_AI_Diagnosis\data\test_list.txt',
        transform=val_transform
    )

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- Load Swin Model ---
    print("⏳ Loading Swin Model...")
    model = models.swin_t(weights=None) # weights=None
    num_ftrs = model.head.in_features
    model.head = nn.Linear(num_ftrs, 14) # <--- Swin-ը 'head'

    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    y_true = []
    y_pred = []

    print("🚀 Starting Prediction...")
    with torch.no_grad():
        for images, labels in tqdm(test_loader):
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.sigmoid(outputs)

            y_true.append(labels.cpu().numpy())
            y_pred.append(probs.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    print("\n" + "="*40)
    print(f"{'DISEASE':<20} | {'AUC SCORE'}")
    print("="*40)

    aucs = []
    for i, label in enumerate(LABELS):
        try:
            auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(auc)
            print(f"{label:20} | {auc:.4f}")
        except ValueError:
            print(f"{label:20} | Error")

    print("-" * 40)
    print(f"🏆 AVERAGE AUC: {np.mean(aucs):.4f}")
    print("=" * 40)

if __name__ == "__main__":
    evaluate_model()

✅ Evaluating Swin Transformer on device: cpu
⏳ Loading Swin Model...
🚀 Starting Prediction...


100%|██████████| 1600/1600 [47:57<00:00,  1.80s/it]



DISEASE              | AUC SCORE
Atelectasis          | 0.7521
Cardiomegaly         | 0.8591
Consolidation        | 0.7339
Edema                | 0.8292
Effusion             | 0.8098
Emphysema            | 0.8964
Fibrosis             | 0.7920
Hernia               | 0.8380
Infiltration         | 0.6935
Mass                 | 0.7713
Nodule               | 0.7262
Pleural_Thickening   | 0.7382
Pneumonia            | 0.6996
Pneumothorax         | 0.8467
----------------------------------------
🏆 AVERAGE AUC: 0.7847
